In [15]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import timm
import os
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

# Team name (replace with your details)
TEAM_NAME = "Vaibhav_13"  # Example: [TeamLeader'sName_HostelNo.]

# Image size for ViT
image_size = 224

# Device setup
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [16]:
# Enhanced augmentations for training (allowed per rules)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test/validation transform
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [17]:
class BirdDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = int(self.data.iloc[idx]['label']) - 1  # Shift 1–200 to 0–199
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

class TestBirdDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.image_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name

In [18]:
# Load and split the Kaggle-provided dataset
df = pd.read_csv('train_labels.csv')  # Provided by Kaggle
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)

train_dataset = BirdDataset(csv_file='train_split.csv', transform=train_transform)
val_dataset = BirdDataset(csv_file='val_split.csv', transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

test_dir = 'data/test'  # Replace with Kaggle test folder path
test_dataset = TestBirdDataset(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

In [19]:
# Load ViT from scratch (no pretrained weights)
model = timm.create_model('vit_large_patch16_224', pretrained=False, num_classes=200)

# Add dropout for regularization
model.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.head.in_features, 200)
)

model = model.to(device)

In [20]:
# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [21]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=5):
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # Training phase
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        scheduler.step()

        # Validation phase with metrics
        model.eval()
        correct = 0
        total = 0
        all_labels = []
        all_probs = []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        # Normalized multi-class accuracy (primary metric)
        val_acc = correct / total
        print(f"Train Loss: {running_loss / len(train_loader.dataset):.4f} | Normalized Accuracy: {val_acc:.4f}")

        # Additional metrics for bonus points
        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)
        auc_score = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
        mean_avg_precision = precision_score(all_labels, np.argmax(all_probs, axis=1), average='macro')
        cm = confusion_matrix(all_labels, np.argmax(all_probs, axis=1))
        sensitivity = np.diag(cm) / np.maximum(np.sum(cm, axis=1), 1)  # Per-class sensitivity
        specificity = []
        for i in range(200):
            tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
            fp = cm[:, i].sum() - cm[i, i]
            specificity.append(tn / max(tn + fp, 1))
        avg_sensitivity = np.mean(sensitivity)
        avg_specificity = np.mean(specificity)

        print(f"AUC: {auc_score:.4f} | Mean Avg Precision: {mean_avg_precision:.4f}")
        print(f"Avg Sensitivity: {avg_sensitivity:.4f} | Avg Specificity: {avg_specificity:.4f}")

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), f'best_model_{TEAM_NAME}.pth')
            print("✅ Saved new best model")
        else:
            patience_counter += 1
            print(f"⚠️ Early stopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("⏹️ Early stopping triggered.")
                break

    return best_acc

In [23]:
# Train the model
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=5)

# Generate submission
model.load_state_dict(torch.load(f'best_model_{TEAM_NAME}.pth'))
model.eval()
submission_df = predict_test_images(model, test_loader, device)
submission_df.to_csv(f'submission_{TEAM_NAME}.csv', index=False)
print(f"Final Normalized Accuracy: {best_acc:.4f}")


Epoch 1/50
Train Loss: 5.5938 | Normalized Accuracy: 0.0048
AUC: 0.5837 | Mean Avg Precision: 0.0000
Avg Sensitivity: 0.0045 | Avg Specificity: 0.9950


/home/vaibhavb/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved new best model

Epoch 2/50
Train Loss: 5.4924 | Normalized Accuracy: 0.0053
AUC: 0.6011 | Mean Avg Precision: 0.0001
Avg Sensitivity: 0.0056 | Avg Specificity: 0.9950


/home/vaibhavb/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Saved new best model

Epoch 3/50


KeyboardInterrupt: 

In [22]:
def predict_test_images(model, test_loader, device):
    model.eval()
    predictions = []
    image_ids = []
    
    with torch.no_grad():
        for images, img_names in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            predicted = predicted.cpu().numpy() + 1  # Shift 0–199 to 1–200
            predictions.extend(predicted)
            image_ids.extend(img_names)
    
    submission_df = pd.DataFrame({'ID': image_ids, 'label': predictions})
    return submission_df

# Load best model and predict
model.load_state_dict(torch.load(f'best_model_{TEAM_NAME}.pth'))
model.to(device)
submission_df = predict_test_images(model, test_loader, device)
submission_df.to_csv(f'submission_{TEAM_NAME}.csv', index=False)
print(f"✅ Submission file 'submission_{TEAM_NAME}.csv' generated!")
print(submission_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'best_model_Vaibhav_13.pth'